## Note:

This notebook analyzes inter-subject phase synchrony (ISPS) using the phase-angle time series extracted from subject-specific ROIs. 

It first computes within-age-bin ISPS, then identifies time segments in which ISPS changes with age, and finally tests whether those age-increasing ISPS segments are associated with lexical surprisal features.

The analysis script here uses *Despicable Me* (DM) as an example. 
The same pipeline is applied to *The Present* (TP) by changing the input files.


## 1. compute inter-subject phase synchrony (ISPS) within age bins

In [ ]:
import numpy as np
import pandas as pd
import os

# ---------------- Configurations ----------------
PHASE_FILE = "movieDM_Whole_CoreLanguage_sstop10perc_ROI_meantimecourse_phase.npy"
DATA_DIR = "movieDM_subdata/"  # used to reconstruct subject order
AGE_CSV = "~/movieDM_subinfo.csv"

SUBJECT_ID_COL = "ID"  
AGE_COL = "MRI_Track,Age_at_Scan" 
N_GROUPS = 17                       # number of age groups
GROUPING = "quantile"              # "quantile" or "equalwidth"
OUTPUT_DIR = "6_movieDM_ISPS/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --------------- Helpers ----------------
def get_subject_list_from_data_dir(data_dir):
    subs = sorted([f.split('_')[0].replace('sub-','') for f in os.listdir(data_dir) if f.endswith(".hdf")])
    return subs  

def compute_group_isps_for_roi(phase_roi):  # phase_roi: (T, Sg)
    """
    Compute pairwise mean synchrony within a group:
    ISPS(t) = mean_{i<j}[ 1 - sin(|phi_i(t) - phi_j(t)| / 2) ]
    Handles NaNs (pairs with any NaN are ignored at that t).
    Returns: (T,)
    """
    T, Sg = phase_roi.shape
    if Sg < 2:
        return np.full(T, np.nan, dtype=np.float32)

    valid = ~np.isnan(phase_roi)  # (T, Sg)
    
    # Build pairwise differences via broadcasting: (T, Sg, Sg)
    diffs = phase_roi[:, :, None] - phase_roi[:, None, :]
    sync = 1.0 - np.sin(np.abs(diffs) / 2.0)  # (T, Sg, Sg)

    # Validity mask for pairs (exclude self-pairs)
    pair_valid = (valid[:, :, None] & valid[:, None, :])
    np.fill_diagonal(pair_valid[0], False)  
    for t in range(T):
        np.fill_diagonal(pair_valid[t], False)

    # Use upper-triangle (i<j) to avoid double-count
    triu_mask = np.triu(np.ones((Sg, Sg), dtype=bool), k=1)  # (Sg, Sg)
    pair_mask = pair_valid & triu_mask[None, :, :]

    # Sum and count per timepoint
    sync = sync.astype(np.float32, copy=False)
    sync[~pair_mask] = np.nan
    
    # Sum/count ignoring NaNs
    num = np.nansum(sync, axis=(1, 2))           # (T,)
    den = np.sum(pair_mask, axis=(1, 2)).astype(np.float32)  # (T,)

    out = np.full(T, np.nan, dtype=np.float32)
    good = den > 0
    out[good] = num[good] / den[good]
    return out

# --------------- Load data ----------------
phase = np.load(PHASE_FILE)  # (T, R, S)
T, R, S = phase.shape
print(f"Loaded phase: T={T}, R={R}, S={S}")

subjects = get_subject_list_from_data_dir(DATA_DIR)
assert len(subjects) == S, f"Subject count mismatch: from DATA_DIR={len(subjects)} vs phase S={S}"

# Load age info and align to phase subject order
demo = pd.read_csv(AGE_CSV)
demo = demo[[SUBJECT_ID_COL, AGE_COL]].dropna()

# Build series of ages in the (S,) order used by phase
age_map = dict(zip(demo[SUBJECT_ID_COL].astype(str), demo[AGE_COL].astype(float)))
ages = np.array([age_map.get(s, np.nan) for s in subjects], dtype=float)
if np.isnan(ages).any():
    missing = np.sum(np.isnan(ages))
    print(f"Warning: {missing} subjects have missing ages; they will be excluded from their groups.")

# --------------- Define age groups ----------------
valid_sub_idx = np.where(~np.isnan(ages))[0]
valid_ages = ages[valid_sub_idx]

if GROUPING == "quantile":
    # Equal-count groups via quantiles
    bins = pd.qcut(valid_ages, q=N_GROUPS, labels=False, duplicates='drop')
    # bins is length len(valid_sub_idx)
    group_labels = np.full(S, -1, dtype=int)  # -1 for missing age
    group_labels[valid_sub_idx] = np.asarray(bins)

elif GROUPING == "equalwidth":
    # Equal-width bins
    group_labels = np.full(S, -1, dtype=int)
    edges = np.linspace(np.nanmin(ages), np.nanmax(ages), N_GROUPS + 1)
    group_labels[valid_sub_idx] = np.digitize(valid_ages, edges[1:-1], right=False)
else:
    raise ValueError("GROUPING must be 'quantile' or 'equalwidth'")

# For each group, collect subject indices and compute stats
group_sub_idx = []
group_stats = []
for g in range(N_GROUPS):
    idx = np.where(group_labels == g)[0]
    group_sub_idx.append(idx)
    mean_age = float(np.nanmean(ages[idx])) if len(idx) > 0 else np.nan
    group_stats.append({"group": g, "n_subjects": int(len(idx)), "mean_age": mean_age})

group_stats_df = pd.DataFrame(group_stats)
group_stats_df.to_csv(os.path.join(OUTPUT_DIR, "age_groups_summary.csv"), index=False)
print("Saved age group summary:", os.path.join(OUTPUT_DIR, "age_groups_summary.csv"))
print(group_stats_df)

# --------------- Compute (T, R, G) ISPS within groups ----------------
# Initialize output
group_isps = np.full((T, R, N_GROUPS), np.nan, dtype=np.float32)

for r in range(R):
    if (r + 1) % 1 == 0:
        print(f"ROI {r+1}/{R}")
    for g in range(N_GROUPS):
        idx = group_sub_idx[g]
        if len(idx) < 2:
            continue  # Need at least 2 subjects to compute ISPS
        phase_roi = phase[:, r, idx]  # (T, Sg)
        group_isps[:, r, g] = compute_group_isps_for_roi(phase_roi)

# Save (T, R, G)
npy_trrg = os.path.join(OUTPUT_DIR, "Whole_CoreLanguage_groupISPS_TRxROIxG.npy")
np.save(npy_trrg, group_isps)
print("Saved (T,R,G) group ISPS:", npy_trrg)

# --------------- Average across ROI -> (T, G) ----------------
net_isps = np.nanmean(group_isps, axis=1)  # (T, G)
npy_tg = os.path.join(OUTPUT_DIR, "Whole_CoreLanguage_groupISPS_TRxG.npy")
np.save(npy_tg, net_isps)
print("Saved (T,G) ROI-averaged group ISPS:", npy_tg)

# save csv
tg_df = pd.DataFrame(net_isps, columns=[f"agegrp_{g}" for g in range(N_GROUPS)])
tg_df.insert(0, "TR", np.arange(T))
csv_tg = os.path.join(OUTPUT_DIR, "Whole_CoreLanguage_groupISPS_TRxG.csv")
tg_df.to_csv(csv_tg, index=False)
print("Saved (T,G) CSV:", csv_tg)

## 2. detect age-increasing segments

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Patch

# ==============================
# Configuration
# ==============================
arr_path = "6_movieDM_ISPS/Whole_CoreLanguage_groupISPS_TRxG.npy"
age_csv  = "6_movieDM_ISPS/age_groups_summary.csv"

burn_in_tr = 10   # exclude first 10 TRs
min_len_tr = 5    # keep only segments >= 5 TRs

out_dir  = Path(f"6_movieDM_ISPS/age_ISPS_stats_thresholded_segments_exc{burn_in_tr}_minlen{min_len_tr}")
out_dir.mkdir(parents=True, exist_ok=True)

alpha_fdr = 0.05
save_fig  = True

# Output filenames
stats_csv_name = f"groupISPS_age_stats_signed_burnin{burn_in_tr}_minlen{min_len_tr}.csv"
segs_csv_name  = f"groupISPS_age_segments_burnin{burn_in_tr}_minlen{min_len_tr}.csv"
valid_tr_csv_name = f"groupISPS_validTR_significance_signed_burnin{burn_in_tr}_minlen{min_len_tr}.csv"

fig_name = f"groupISPS_age_significance_posneg_burnin{burn_in_tr}_minlen{min_len_tr}.png"

# ==============================
# Helpers
# ==============================
def contiguous_segments(mask_bool: np.ndarray):
    segs, in_seg, start = [], False, None
    for i, m in enumerate(mask_bool):
        if m and not in_seg:
            in_seg, start = True, i
        elif (not m) and in_seg:
            in_seg = False
            segs.append((start, i - 1))
    if in_seg:
        segs.append((start, len(mask_bool) - 1))
    return segs

def filter_by_minlen(segs, min_len: int):
    return [(s, e) for (s, e) in segs if (e - s + 1) >= min_len]

def summarize_segments(segments, label, betas, tvals, rvals, p_fdr):
    rows = []
    for s, e in segments:
        sl = slice(s, e + 1)
        length = e - s + 1
        p_min = float(np.nanmin(p_fdr[sl]))
        p_max = float(np.nanmax(p_fdr[sl]))
        beta_mean = float(np.nanmean(betas[sl]))
        r_mean = float(np.nanmean(rvals[sl]))

        local_idx = int(np.nanargmax(np.abs(tvals[sl])))
        peak_idx = s + local_idx

        rows.append({
            "band_sign": label,
            "start_timepoint": int(s),
            "end_timepoint": int(e),
            "length": int(length),
            "peak_timepoint": int(peak_idx),
            "peak_t_abs": float(np.abs(tvals[peak_idx])),
            "peak_beta": float(betas[peak_idx]),
            "peak_r_partial": float(rvals[peak_idx]),
            "min_p_fdr": p_min,
            "max_p_fdr": p_max,
            "mean_beta": beta_mean,
            "mean_r_partial": r_mean
        })
    return pd.DataFrame(rows)

# ==============================
# Load data
# ==============================
Y = np.load(arr_path)  # (T, G)
T, G = Y.shape
print(f"Loaded ISPS array: {Y.shape}")

age_df = pd.read_csv(age_csv)
ages = age_df["mean_age"].to_numpy(dtype=float)
if len(ages) != G:
    raise ValueError(f"Mismatch: Y has G={G} groups but age_csv has {len(ages)} rows.")

print(f"Loaded {G} age groups with mean ages: {np.round(ages, 2)}")

# Burn-in mask
valid_time = np.zeros(T, dtype=bool)
valid_time[burn_in_tr:] = True

# ==============================
# Fit OLS: ISPS_t ~ mean_age
# ==============================
betas = np.full(T, np.nan, dtype=float)
tvals = np.full(T, np.nan, dtype=float)
pvals = np.full(T, np.nan, dtype=float)
rvals = np.full(T, np.nan, dtype=float)

X = sm.add_constant(ages)

for t in range(T):
    if not valid_time[t]:
        continue
    y_t = Y[t, :]
    if np.isnan(y_t).any() or np.isnan(ages).any():
        continue

    res = sm.OLS(y_t, X).fit()

    beta_age = res.params[1]
    t_age = res.tvalues[1]
    p_age = res.pvalues[1]
    df_resid = res.df_resid
    r_partial = t_age / np.sqrt(t_age**2 + df_resid)

    betas[t] = beta_age
    tvals[t] = t_age
    pvals[t] = p_age
    rvals[t] = r_partial

# ==============================
# FDR across valid timepoints only
# ==============================
p_fdr = np.full(T, np.nan, dtype=float)
sig_mask = np.zeros(T, dtype=bool)

valid_idx = np.where(valid_time & ~np.isnan(pvals))[0]
if len(valid_idx) == 0:
    raise RuntimeError("No valid timepoints for FDR correction (check burn_in_tr / NaNs).")

rej, p_fdr_valid, _, _ = multipletests(pvals[valid_idx], alpha=alpha_fdr, method="fdr_bh")
p_fdr[valid_idx] = p_fdr_valid
sig_mask[valid_idx] = rej.astype(bool)

pos_mask_raw = sig_mask & (betas > 0)
neg_mask_raw = sig_mask & (betas < 0)

# ==============================
# Segments + min length filter
# ==============================
pos_segs = filter_by_minlen(contiguous_segments(pos_mask_raw), min_len_tr)
neg_segs = filter_by_minlen(contiguous_segments(neg_mask_raw), min_len_tr)

# Build "real" (kept) masks from the kept segments
pos_mask = np.zeros(T, dtype=bool)
neg_mask = np.zeros(T, dtype=bool)
for s, e in pos_segs:
    pos_mask[s:e+1] = True
for s, e in neg_segs:
    neg_mask[s:e+1] = True

# Signed significance label you can reuse downstream:
#   +1 = kept positive segments
#   -1 = kept negative segments
#    0 = valid timepoints not in kept segments
#   NaN = excluded burn-in
significance_signed = np.full(T, np.nan, dtype=float)
significance_signed[valid_time] = 0
significance_signed[pos_mask] = 1
significance_signed[neg_mask] = -1

# Segment summaries
pos_df = summarize_segments(pos_segs, "positive", betas, tvals, rvals, p_fdr)
neg_df = summarize_segments(neg_segs, "negative", betas, tvals, rvals, p_fdr)
segs_df = pd.concat([pos_df, neg_df], ignore_index=True)

# ==============================
# Save outputs
# ==============================
# Per-TR stats + signed label
out_df = pd.DataFrame({
    "TR": np.arange(T),
    "valid_timepoint": valid_time.astype(int),
    "beta_age": betas,
    "t_age": tvals,
    "r_partial_age": rvals,
    "p_age_raw": pvals,
    "p_age_fdr_bh": p_fdr,
    "significant_fdr05_raw": (pos_mask_raw | neg_mask_raw).astype(int),
    "significant_pos_fdr05_raw": pos_mask_raw.astype(int),
    "significant_neg_fdr05_raw": neg_mask_raw.astype(int),
    "significance_signed": significance_signed, 
})

out_df.to_csv(out_dir / stats_csv_name, index=False)
segs_df.to_csv(out_dir / segs_csv_name, index=False)

valid_tr_df = out_df.loc[valid_time, ["TR", "significance_signed"]].copy()
valid_tr_df.to_csv(out_dir / valid_tr_csv_name, index=False)

print("Saved:")
print("  ", out_dir / stats_csv_name)
print("  ", out_dir / segs_csv_name)
print("  ", out_dir / valid_tr_csv_name)

# ==============================
# Plot
# ==============================
mean_tc = np.nanmean(Y, axis=1)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(np.arange(T), mean_tc, lw=1.5, color="k")
ax.set_xlabel("Timepoint (TR)", size=13)
ax.set_ylabel("Mean ISPS (across age groups)", size=14)

for s, e in pos_segs:
    ax.axvspan(s, e, color="orange", alpha=0.35)
for s, e in neg_segs:
    ax.axvspan(s, e, color="skyblue", alpha=0.35)

# burn-in region shading
if burn_in_tr > 0:
    ax.axvspan(0, burn_in_tr - 1, color="gray", alpha=0.15)

valid_count = int(valid_time.sum())
pos_count = int(np.sum(pos_mask & valid_time))
neg_count = int(np.sum(neg_mask & valid_time))
no_count  = valid_count - (pos_count + neg_count)

pos_pct = 100 * pos_count / valid_count
neg_pct = 100 * neg_count / valid_count
no_pct  = 100 * no_count  / valid_count

ax.legend(handles=[
    Patch(facecolor="orange", alpha=0.35, edgecolor="none",
          label=f"Positive (kept segs) ({pos_pct:.1f}%, FDR<.05)"),
    Patch(facecolor="skyblue", alpha=0.35, edgecolor="none",
          label=f"Negative (kept segs) ({neg_pct:.1f}%, FDR<.05)"),
    Patch(facecolor="white", edgecolor="black", linestyle="--",
          linewidth=0.4, label=f"No relation ({no_pct:.1f}%)"),
    Patch(facecolor="gray", alpha=0.15, edgecolor="none",
          label=f"Excluded burn-in (first {burn_in_tr} TR)")
], loc=(0.67, 1.02), frameon=False)

plt.tight_layout()

fig_path = out_dir / fig_name
if save_fig:
    plt.savefig(fig_path, dpi=200)
    print("Saved figure:", fig_path)

plt.close()

## 3. permutation-based surprisal feature analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ==============================
# Configuration
# ==============================
TIME_COLUMN = "TR"
PEAK_SHIFT_TR = 5

ISPS_PATH = "6_movieDM_ISPS/age_ISPS_stats_thresholded_segments_exc10_minlen5/groupISPS_age_stats_signed_burnin10_minlen5.csv"
FEATURES_PATH = "~/movieDM_lexisurp_10features_lanczos.csv" #lanczos resampled movie features

OUTDIR = f"6_movieDM_ISPS/age_ISPS_stats_thresholded_segments_exc10_minlen5/spoken_words_permutation_peak_surprisal_shift{PEAK_SHIFT_TR}_observedlen/"
os.makedirs(OUTDIR, exist_ok=True)

SURPRISAL_COL = "GPT3_surprisal"  

SEED = 42

# ==============================
# Helpers
# ==============================
def contiguous_runs(tr_values):
    tr_values = np.asarray(tr_values, dtype=int)
    if tr_values.size == 0:
        return []
    runs = []
    start = tr_values[0]
    prev = tr_values[0]
    for t in tr_values[1:]:
        if t == prev + 1:
            prev = t
        else:
            runs.append((start, prev))
            start = t
            prev = t
    runs.append((start, prev))
    return runs

def extract_segment_values(df, start_tr, end_tr, value_col, tr_col=TIME_COLUMN):
    seg = df.loc[(df[tr_col] >= start_tr) & (df[tr_col] <= end_tr), value_col].to_numpy()
    return seg

def sample_onsets(n_onsets, valid_onsets, seg_len, allow_overlap=True, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    valid_onsets = np.asarray(valid_onsets, dtype=int)
    if allow_overlap:
        return rng.choice(valid_onsets, size=n_onsets, replace=False)  # 不重复 onset
    else:
        max_tries = 2000
        for _ in range(max_tries):
            onsets = rng.choice(valid_onsets, size=n_onsets, replace=False)
            onsets.sort()
            ok = True
            prev_end = -np.inf
            for o in onsets:
                if o <= prev_end:
                    ok = False
                    break
                prev_end = o + seg_len - 1
            if ok:
                return onsets
        raise RuntimeError("Cannot sample enough segments")

# ==============================
# Load & align
# ==============================
ISPS_df = pd.read_csv(ISPS_PATH)
Features_df = pd.read_csv(FEATURES_PATH)

if TIME_COLUMN not in ISPS_df.columns or TIME_COLUMN not in Features_df.columns:
    raise ValueError("TR column missing in input files.")
if "significance_signed" not in ISPS_df.columns:
    raise ValueError("ISPS file missing significance_signed.")

ISPS_df[TIME_COLUMN] = pd.to_numeric(ISPS_df[TIME_COLUMN], errors="coerce").astype("Int64")
Features_df[TIME_COLUMN] = pd.to_numeric(Features_df[TIME_COLUMN], errors="coerce").astype("Int64")

# remove _lanczos suffix
Features_df = Features_df.copy()
Features_df.rename(columns=lambda c: c.replace("_lanczos", ""), inplace=True)

ISPS_use = ISPS_df[[TIME_COLUMN, "significance_signed"]].copy()

merged = pd.merge(
    Features_df[[TIME_COLUMN, SURPRISAL_COL]],
    ISPS_use,
    on=TIME_COLUMN,
    how="inner",
    validate="one_to_one"
).sort_values(TIME_COLUMN).reset_index(drop=True)

# shift features backward by PEAK_SHIFT_TR to align with the "peak" timepoints of ISPS segments
merged["label_shifted"] = merged["significance_signed"].shift(-PEAK_SHIFT_TR)

# keep only valid labels
valid_mask = merged["label_shifted"].isin([-1, 0, 1])
merged_valid = merged.loc[valid_mask].reset_index(drop=True)

# ==============================
# Step 1: get label_shifted==1 segments; remove segments with surprisal variance==0
# ==============================
pos_trs = merged_valid.loc[merged_valid["label_shifted"] == 1, TIME_COLUMN].dropna().astype(int).to_numpy()
pos_trs.sort()

runs = contiguous_runs(pos_trs)  

segments = []
for (st, ed) in runs:
    seg_vals = extract_segment_values(merged_valid, st, ed, SURPRISAL_COL, TIME_COLUMN)
    # check if variance==0 
    seg_var = float(np.var(seg_vals)) if seg_vals.size > 0 else np.nan
    keep = (seg_vals.size > 0) and (seg_var > 0.0)
    segments.append({
        "start_tr": int(st),
        "end_tr": int(ed),
        "len_tr": int(ed - st + 1),
        "surprisal_var": seg_var,
        "keep": bool(keep),
    })

seg_df = pd.DataFrame(segments)
seg_df.to_csv(os.path.join(OUTDIR, "pos_segments_with_variance_filter.csv"), index=False)

kept_seg_df = seg_df[seg_df["keep"]].copy()
X = int(len(kept_seg_df))
avg_len_tr = float(kept_seg_df["len_tr"].mean()) if X > 0 else np.nan

summary = {
    "n_pos_segments_total": int(len(seg_df)),
    "n_pos_segments_kept": X,
    "avg_len_tr_kept": avg_len_tr,
    "median_len_tr_kept": float(kept_seg_df["len_tr"].median()) if X > 0 else np.nan,
}
pd.DataFrame([summary]).to_csv(os.path.join(OUTDIR, "summary_step1.csv"), index=False)

print("Step1 summary:", summary)

if X == 0:
    raise RuntimeError("No segments remained, please check!")

# Determine SEG_LEN for permutation based on observed average length of kept segments.
SEG_LEN = int(np.round(avg_len_tr))
SEG_LEN = max(1, SEG_LEN)
print(f"Using SEG_LEN={SEG_LEN} TR (rounded from avg_len_tr={avg_len_tr:.3f}).")

# ==============================
# Step 2: observed mean of segment maxima
# ==============================
obs_maxima = []
for _, r in kept_seg_df.iterrows():
    st, ed = int(r["start_tr"]), int(r["end_tr"])
    seg_vals = extract_segment_values(merged_valid, st, ed, SURPRISAL_COL, TIME_COLUMN)
    obs_maxima.append(float(np.max(seg_vals)))

obs_maxima = np.asarray(obs_maxima, dtype=float)
observed = float(np.mean(obs_maxima))

pd.DataFrame({"segment_max_surprisal": obs_maxima}).to_csv(
    os.path.join(OUTDIR, "observed_segment_maxima.csv"), index=False
)

print(f"Observed mean of segment maxima = {observed:.6f} (X={X})")

# ==============================
# Step 3: permutation using observed length distribution + variance>0 filtering
# ==============================
rng = np.random.default_rng(SEED)

merged_valid_sorted = merged_valid.sort_values(TIME_COLUMN).reset_index(drop=True)
surp_series = merged_valid_sorted[SURPRISAL_COL].to_numpy(dtype=float)
tr_series = merged_valid_sorted[TIME_COLUMN].to_numpy(dtype=int)

N = len(tr_series)

# observed length distribution (one length per kept segment)
obs_lens = kept_seg_df["len_tr"].astype(int).to_numpy()
if len(obs_lens) != X:
    raise RuntimeError("obs_lens length mismatch with X. Check kept_seg_df / X.")

EPS = 1e-12
N_PERM = 10000 
MAX_TRIES_PER_SEG = 5000  #max attempts to find a valid window for each segment in each permutation
ALLOW_OVERLAP = True      #whether to allow sampled segments to overlap in time (if False, will check and resample if overlap)

perm_stats = np.zeros(N_PERM, dtype=float)

for b in range(N_PERM):
   
    lens = obs_lens.copy()
    rng.shuffle(lens)

    chosen_max = []
    chosen_intervals = []  

    for L in lens:
        if L <= 0:
            raise RuntimeError(f"Invalid segment length L={L} in obs_lens.")
        if L > N:
            raise RuntimeError(f"Observed segment length L={L} exceeds data length N={N}.")

        valid_onset_idx = np.arange(0, N - L + 1)

        ok = False
        for _ in range(MAX_TRIES_PER_SEG):
            oi = int(rng.choice(valid_onset_idx, size=1, replace=False)[0])
            start = oi
            end = oi + L - 1

            if not ALLOW_OVERLAP:
                overlaps = any(not (end < s or start > e) for (s, e) in chosen_intervals)
                if overlaps:
                    continue

            window = surp_series[start:start + L]
            if float(np.var(window)) <= EPS:
                continue

            chosen_max.append(float(np.max(window)))
            if not ALLOW_OVERLAP:
                chosen_intervals.append((start, end))
            ok = True
            break

        if not ok:
            raise RuntimeError(f"Permutation {b}: Cannot find valid window of length L={L} with variance>0 after {MAX_TRIES_PER_SEG} tries. ")

    perm_stats[b] = float(np.mean(chosen_max))

# calculate permutation-based p-values
observed = float(observed)  
p_right = (np.sum(perm_stats >= observed) + 1) / (N_PERM + 1)
p_two = (np.sum(np.abs(perm_stats - np.mean(perm_stats)) >= np.abs(observed - np.mean(perm_stats))) + 1) / (N_PERM + 1)

# save
pd.DataFrame({"perm_mean_of_segment_max": perm_stats}).to_csv(
    os.path.join(OUTDIR, f"perm_distribution_lenMatched_N{N_PERM}.csv"),
    index=False
)
pd.DataFrame([{
    "observed_mean_of_segment_max": observed,
    "p_right_tail": p_right,
    "p_two_sided_centered": p_two,
    "n_perm": N_PERM,
    "X_segments": X,
    "lengths_used": "observed_len_tr_distribution",
    "variance_filter_eps": EPS,
    "allow_overlap": ALLOW_OVERLAP,
    "seed": SEED,
}]).to_csv(os.path.join(OUTDIR, "permutation_test_results_lenMatched.csv"), index=False)

print(f"Permutation (len-matched) results: p_right={p_right:.6g}, p_two={p_two:.6g}")

# plot
plt.figure(figsize=(7, 4))
plt.hist(perm_stats, bins=50, alpha=0.8)
plt.axvline(observed, linewidth=2)
plt.xlabel("Permutation: mean(max surprisal per segment)")
plt.ylabel("Count")
plt.title(f"Len-matched permutation (N={N_PERM}), observed={observed:.4f}\n"
          f"X={X}, p_right={p_right:.4g}")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "perm_histogram_lenMatched.png"), dpi=200)
plt.close()